# Введение в рекомендательные системы

_Основная цель — собрать необходимый минимум знаний для построения рекомендательной системы в одном месте._

### Примеры рекомендательных систем

У социальных сетей возникают следующие задачи:

* Персональные ленты
* Люди, с которыми вы можете быть знакомы

Но вообще ранжировать можно что угодно:

* Поисковую выдачу
* Адреса назначения в агрегаторах такси

### Netflix Prize
Соревнование проводилось почти три года, с _2 октября 2006 по 26 июня
2009 года_.

**Основная цель** — превзойти модель Netflix на 10% по RMSE на данных о
пользовательских рейтингах фильмов

**Призы** — 1 миллион долларов за первое место.

_Представители компании шутили, что всего за 1 миллион долларов на них работало множество компаний со всего мира._ 

**Краткие итоги**:
* "Бум" в исследовательской среде. Многие методы из этого
соревнования до сих пор используются.
* В "прод" ушла модель не с первого места. Классика kaggle vs prod.
* Ошибка с метрикой. Модели в итоге использовались для
ранжирования фильмов, а не для восстановления рейтингов как
таковых. 

# Блок 1. Постановка задачи и работа с данными

* Задача сводится к ранжированию объектов или предсказанию оценки $r_{ui}$ для пользователя $u$ и объекта $i$.
* Данные для обучения бывают двух видов:
    * Явные (Explicit): оценки или покупки. Этому таргету можно доверять, но таких данных обычно мало.
    * Неявные (Implicit): клики или просмотры. Они не говорят явно о том, как пользователь оценил объект, но таких данных в системе гораздо больше.
* Базовый датасет идеально ложится в `pandas.DataFrame` и содержит идентификатор пользователя, идентификатор объекта, и опционально — оценку и время

In [35]:
import pandas as pd

path = '../data/raw/train_data/part_000.parquet'
df = pd.read_parquet(path)

In [36]:
df.head()

,timestamp,eid,user_id,item_id
0,1768964391000,7,0,11148055
1,1768964405000,7,0,164404551
2,1768964450000,7,0,164404551
3,1768964470000,4,0,164404551
4,1768964486000,7,0,164404551


Тут у нас представленно неявное взаимодействие, так как мы наверняка не знаем, что за интеракция произошла между пользователем и айтемом.

# Блок 2. Оптимизация памяти в Pandas

* При выгрузке логов в pandas часто возникают проблемы с оперативной памятью.
* Идентификаторы-строки лучше приводить к `CategoricalDType`.
* Численные колонки с пропусками (которые `pandas` конвертирует во `float`) — к `IntegerDType`.
* Это позволяет сэкономить от 20% до 80% памяти

In [37]:
df.info() # memory_usage='deep' необходим для вычисления точного размеров объектов типа 

<class 'pandas.DataFrame'>
RangeIndex: 51682001 entries, 0 to 51682000
Data columns (total 4 columns):
 #   Column     Dtype 
---  ------     ----- 
 0   timestamp  int64 
 1   eid        uint32
 2   user_id    uint32
 3   item_id    uint32
dtypes: int64(1), uint32(3)
memory usage: 985.8 MB


Почти 1 ГБ для одной части данных — это немало! Создатели датасета уже немного оптимизировали его, сохранив идентификаторы как `uint32` вместо стандартного для python `int64`. В нашем случае радикального экономия памяти мы не увидем, однако перевод в категориальный тип нам очень пригодится для следующего шага.

In [38]:
df['user_id'] = df['user_id'].astype('category')
df['item_id'] = df['item_id'].astype('category')

In [39]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51682001 entries, 0 to 51682000
Data columns (total 4 columns):
 #   Column     Dtype   
---  ------     -----   
 0   timestamp  int64   
 1   eid        uint32  
 2   user_id    category
 3   item_id    category
dtypes: category(2), int64(1), uint32(1)
memory usage: 1.0 GB


* использование категориальных типов позволяет экономить от 20 до 80 процентов памяти. Однако это правило работает, когда мы конвертируем "тяжелые" строки или тип `int64`. Наши данные уже были оптимально сжаты в `uint32`. Тип `category` работает как словарь: он создает массив индексов и отдельную структуру с уникальными значениями. Поскольку оригинальные числа и так были маленькими, добавление этого "словаря" немного увеличило общий размер.

# Блок 3. Переход от DataFrame к разреженным матрицам

* Лог `(user, item, rating)` в `pandas` — это лишь представление матрицы, но библиотеки рекомендательных систем принимают на вход разреженные матрицы из модуля `scipy.sparse`.
* Для создания матриц используется формат coo_matrix, а для оптимизированных математических операций — `csr_matrix` или `csc_matrix`.  

In [40]:
# если передадим user_id: 10, 50, 1000 coo_matrix создаст огромную матрицу из 10001 строки
# потому давайте сожмем id и уберём пропуски
user_codes = df['user_id'].cat.codes
item_codes = df['item_id'].cat.codes

Теперь у нас есть массивы непрерывных числовых индексов, которые можно использовать как координаты.

In [41]:
import numpy as np
from scipy.sparse import coo_matrix

data = np.ones(len(df))

# (data, (row, col))
interaction_matrix = coo_matrix((data, (user_codes, item_codes)))

In [42]:
interaction_matrix_src = interaction_matrix.tocsr()

# Блок 4. Виды рекомендательных систем и Бейзлайны

* Рекомендательные системы глобально делятся на неперсонализированные и персонализированные.
* Самый простой неперсонализированный подход — рекомендовать "Популярное" (глобально, по категориям или группам)

In [43]:
df['item_id'].value_counts().head(5)

item_id
2787850      2652
22577334     2032
128186904    1472
92072874      915
103113575     776
Name: count, dtype: int64

Вот и наш бэйзлайн рекомендаций :D

Давайте протестируем на следующих метриках

Пусть:

* $U$ — множество всех пользователей в тестовой выборке.
* $T_u$ — множество реальных товаров (target/ground truth), с которыми взаимодействовал пользователь $u$.
* $R_u@k$ — множество из $k$ предсказанных (рекомендованных) товаров для пользователя $u$.

**1. HitRate@k (Доля попаданий)**
Эта метрика отвечает на вопрос: "Для какой доли пользователей мы угадали хотя бы один товар?".

$$
HitRate@k = \frac{1}{|U|} \sum_{u \in U} \mathbb{1}_{\{|R_u@k \cap T_u| > 0\}}
$$

**2. Recall@k (Полнота)**
Эта метрика отвечает на вопрос: "Какую долю из всех реальных предпочтений пользователя мы смогли покрыть нашими $k$ рекомендациями?".

$$
Recall@k = \frac{1}{|U|} \sum_{u \in U} \frac{|R_u@k \cap T_u|}{|T_u|}
$$


In [44]:
top_160_items = df['item_id'].value_counts()[:160].index.to_list()

In [45]:
path_eval = '../data/raw/local_eval.csv'
df_truth = pd.read_csv(path_eval)

In [46]:
total_users = df_truth['user_id'].nunique()

df_hits = df_truth[df_truth['item_id'].isin(top_160_items)]

hit_users = df_hits['user_id'].nunique()

hit_rate_160 = hit_users / total_users

In [47]:
total_items_per_user = df_truth.groupby('user_id').size()

hits_per_user = df_truth[df_truth['item_id'].isin(top_160_items)].groupby('user_id').size()

recall_160 = (hits_per_user / total_items_per_user).fillna(0).mean()

In [48]:
print(f"Hit Rate@160: {hit_rate_160:.4f}")
print(f"Recall@160: {recall_160:.4f}")

Hit Rate@160: 0.0031
Recall@160: 0.0021


# Блок 5. Контентная фильтрация

* Помимо таргета, можно использовать характеристики (метаданные) объектов и пользователей.
* Основная идея Content-based подхода — использовать характеристики объекта для поиска похожих.
* Функция оценки обычно строится на скалярном произведении или косинусном расстоянии векторов

In [49]:
path = '../data/raw/item_features.parquet'
# df_features = pd.read_parquet(path)

Здесь мы словили первый OOM — out of memory

Чтобы подсмотреть структуру файла и при этом вообще не загружать сами данные в память, мы можем использовать библиотеку `pyarrow`. Она позволяет прочитать только легкие метаданные файла.

In [50]:
import pyarrow.parquet as pq

path = '../data/raw/item_features.parquet'
parquet_file = pq.ParquetFile(path)

# Выводим названия всех колонок
print(parquet_file.schema.names)

['item_id', 'vertical_id', 'category_ext_y', 'region_id_y', 'loc_id_y', 'sid_0_y', 'sid_1_y', 'sid_2_y', 'sid_3_y']


Переходим на `polars` с возможностьюю не загружать всё в озу.

In [51]:
import polars as pl

df_lazy = pl.scan_parquet('../data/raw/train_data/part_000.parquet')
df_features_lazy = pl.scan_parquet('../data/raw/item_features.parquet').select(['item_id', 'category_ext_y'])

In [52]:
df_merged_lazy = df_lazy.join(df_features_lazy, on='item_id', how='inner')

In [53]:
user_category_counts = (
    df_lazy
    .join(df_features_lazy, on='item_id', how='inner')
    .group_by(['user_id', 'category_ext_y'])
    .len()
)

In [54]:
sorted_counts = user_category_counts.sort(
    by=['user_id', 'len'],
    descending=[False, True]
)

In [55]:
top_categories = (
    sorted_counts
    .group_by('user_id')
    .first()
)

In [56]:
item_popularity = (
    df_merged_lazy
    .group_by(['item_id', 'category_ext_y'])
    .len()
    .rename({'len': 'clicks'})
)

In [57]:
item_popularity = item_popularity.sort('clicks', descending=True)

In [58]:
top_items_per_category = (
    item_popularity
    .group_by('category_ext_y')
    .head(160)
)

In [59]:
recommendations = top_categories.join(top_items_per_category, on='category_ext_y', how='left')

In [60]:
final_recommendations = (
    recommendations
    .group_by('user_id')
    .agg(pl.col('item_id'))
)

Давайте теперь посмотрим на `Recall@160`

In [61]:
preds = final_recommendations.rename({'item_id': 'predicted_items'})

df_truth_lazy = pl.scan_csv('../data/raw/local_eval.csv')

recall_160 = (
    df_truth_lazy
    .join(preds, on='user_id', how='left')  # Используем .lazy() для объединения
    .with_columns(
        pl.col('predicted_items').list.contains(pl.col('item_id')).fill_null(False).alias('is_hit')
    )
    .group_by('user_id')
    .agg(
        pl.col('is_hit').mean().alias('user_recall')
    )
    .select(
        pl.col('user_recall').mean().alias('final_macro_recall')
    )
    .collect()
)

print(f"Recall@160: {recall_160['final_macro_recall'][0]:.4f}")

Recall@160: 0.0104


Отлично, как можно заметить, учёт категорий в рекоммендации популярный айтемов повысил нашу метрику в ~ 5 раз.

# Блок 6. Коллаборативная фильтрация

* Идея подхода заключается в использовании истории взаимодействий пользователей с объектами для получения векторных представлений.
* Делится на два базовых типа:
    * `Neighbour-based (Memory-based)`: использует строки или столбцы матрицы оценок (`User-user` или `Item-item` схожесть).
    * `Model-based`: использует матричные разложения ($R \approx P \times Q^T$) для построения внутренних векторных (латентных) представлений.

### Подготовим матрицу интеракций

```python
import scipy.sparse as sp
import numpy as np

# Предположим, у нас есть плоский датафрейм df_train_pandas 
user_ids = df_train_pandas['user_id'].values
item_ids = df_train_pandas['item_id'].values
clicks = np.ones(len(user_ids)) # Если просто клик = 1

# Создаем разреженную матрицу (формат CSR оптимизирован для вычислений)
user_item_matrix = sp.csr_matrix(
    (clicks, (user_ids, item_ids)), 
    shape=(user_ids.max() + 1, item_ids.max() + 1)
)
```

### Alternating Least Squares

```python
import implicit

als_model = implicit.als.AlternatingLeastSquares(
    factors=64, 
    regularization=0.1, 
    iterations=20, 
    random_state=42
)

als_model.fit(user_item_matrix)

# Получаем рекомендации для конкретного пользователя (например, user_id = 123)
recommended_items, scores = als_model.recommend(
    123, 
    user_item_matrix[123], 
    N=160, 
    filter_already_liked_items=True
)

print(f"Рекомендованные товары: {recommended_items}")
```

### Neighbour-based 

```python
from implicit.nearest_neighbours import BM25Recommender

# K - количество соседей, которых мы запоминаем для каждого товара
bm25_model = BM25Recommender(K=100)

bm25_model.fit(user_item_matrix)

recommended_items_knn, scores_knn = bm25_model.recommend(
    123, 
    user_item_matrix[123], 
    N=160,
    filter_already_liked_items=True
)
```

**Важный нюанс: `implicit` работает с сырыми числовыми индексами. Если в ваших данных `user_id` или `item_id` — это огромные числа (например, 100500), размер матрицы получится 100500 x 100500, состоящей из пустых нулей. Перед сборкой матрицы идентификаторы всегда нужно перекодировать в непрерывный ряд от 0 до N-1 (с помощью словарей маппинга или `LabelEncoder`), а после получения рекомендаций — раскодировать обратно.**